In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras.models import load_model
from keras.layers import concatenate, Dense
from tensorflow.keras import regularizers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import Callback, EarlyStopping

## Simulation4b
English   orth--> orth

In [ ]:
X_id  = pd.read_excel('Non_case_Sensitive_word_encodings.xlsx', sheet_name='identity',
                      index_col=0, engine='openpyxl').values.astype('float32')

Y     = pd.read_excel('Non_case_Sensitive_orthograph_encodings.xlsx', sheet_name='orth_vector',
                       engine='openpyxl').values.astype('float32')

X_all= X_id
inp = Input(shape=(X_all.shape[1],), name='in_all')

h1  = Dense(700, activation='sigmoid')(inp)
h2  = Dense(500, activation='sigmoid')(h1)

out = Dense(260, activation='sigmoid')(h2)

model4c = Model(inputs=inp, outputs=out)



model4c.compile(optimizer='adam', loss='mse')
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10)

model4c.fit(X_all, Y, batch_size=25, epochs=700, callbacks=[callback])



model4c.save('simulation4c_english_model.h5')


Epoch 1/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - loss: 0.1186
Epoch 2/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0264
Epoch 3/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0264
Epoch 4/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0262
Epoch 5/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - loss: 0.0262
Epoch 6/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - loss: 0.0257
Epoch 7/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - loss: 0.0257
Epoch 8/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0256
Epoch 9/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.0258
Epoch 10/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.0256
Epoch 11/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0258
Epoch 12/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0256
Epoch 13/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.0256
Epoch 14/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.0256
Epoch 15/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - lo

# Test the model
We will find the correlation distance between the original word and its different modified forms.

In [ ]:
Y     = pd.read_excel('Non_case_Sensitive_orthograph_encodings.xlsx', sheet_name='orth_vector',
                       engine='openpyxl').values.astype('float32')


data_lst= ["Non_case_Sensitive_TL_Same_Case", "Non_case_Sensitive_TL_DF_Case", "Non_case_Sensitive_SL_Same_Case", "Non_case_Sensitive_SL_DF_Case"]
df_sim5 = pd.DataFrame()
for document in data_lst:
    X_id   = pd.read_excel(document+".xlsx", sheet_name='identity', index_col=0).values.astype(np.float32)
    #X_pos  = pd.read_excel(document+".xlsx", sheet_name='position', index_col=0).values.astype(np.float32)
    pred = model4c.predict(X_id)
    valz = []
    for i in range(len(Y)):
      r = np.corrcoef(pred[i], Y[i])[0,1]
      valz.append(round(1-r,4))


    df_sim5[f'target-{document}']= valz

df_sim5.to_excel('simulation4c_correlations_test_3.xlsx',index=False)

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


# Correlation Distance Between Words that share the same stem

In [ ]:
import itertools
import pandas as pd
import numpy as np

# Assuming model4c and X_id are already defined and loaded from previous steps

df_words    = pd.read_excel('words_with_vectors.xlsx', sheet_name='words',     index_col=0)
words    = df_words.iloc[:,0].values
pred_all = model4c.predict(X_id)

df_pred = pd.DataFrame(pred_all, columns=[f'v{i}' for i in range(pred_all.shape[1])])
df_pred['word'] = words

# Load roots file
df_roots = pd.read_excel('words_with_vectors.xlsx', sheet_name='grouped_by_stem',     index_col=0)

# Merge so each row has its vector
df_merged = df_roots.merge(df_pred, on='word', how='inner')

# compute all pairwise distances for each root
rows = []
for root, grp in df_merged.groupby('stem'):
    items = list(zip(grp['word'], grp[[f'v{i}' for i in range(pred_all.shape[1])]].values))
    for (w1, vec1), (w2, vec2) in itertools.combinations(items, 2):
        r = np.corrcoef(vec1, vec2)[0,1]
        dist = 1 - r
        rows.append({'stem': root, 'word1': w1, 'word2': w2, 'distance': dist})

df_pairs = pd.DataFrame(rows)

# Optionally - summary stats per root
summary = df_pairs.groupby('stem')['distance'].agg(
    mean_dist='mean',
    std_dist ='std',
    min_dist ='min',
    max_dist ='max'
).reset_index()

# Save results to Excel
with pd.ExcelWriter('simulation4c_stem_pairwise_distances_3.xlsx') as wr:
    df_pairs.to_excel(wr, sheet_name='all_pairs', index=False)
    summary.to_excel(wr, sheet_name='per_stem_summary', index=False)

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


# Correlation Distance Between the word and other 4 conditions

In [ ]:
import numpy as np
import pandas as pd

df_words    = pd.read_excel('words_with_vectors.xlsx', sheet_name='words',     index_col=0)
words    = df_words.iloc[:,0].values
pred_all = model4c.predict(X_id)



df_pred = pd.DataFrame(pred_all, columns=[f'v{i}' for i in range(pred_all.shape[1])])
df_pred['word'] = words


PAIRS_XLSX  = 'words_with_vectors.xlsx'
PAIRS_SHEET = '4_conditions'


vec_cols = [c for c in df_pred.columns if c.startswith('v')]
vec_map  = dict(zip(df_pred['word'], df_pred[vec_cols].values))

# read pairs
pairs = pd.read_excel(PAIRS_XLSX, sheet_name=PAIRS_SHEET)

def corr_distance(w1, w2):
    v1 = vec_map.get(w1)
    v2 = vec_map.get(w2)
    if v1 is None or v2 is None:
        return np.nan
    r = np.corrcoef(v1, v2)[0, 1]
    return 1 - r

pairs['distance'] = pairs.apply(lambda r: corr_distance(r['word1'], r['word2']), axis=1)

# keep exactly 4 columns in this order
out = pairs[['word1', 'word2', 'condition', 'distance']]
out.to_excel('simulation_4c_4_conditions_pairs_with_distance_3.xlsx', index=False)



15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
